In [ ]:
!pip install librosa scikit-learn soundfile numpy

In [ ]:
import os
import numpy as np
import librosa

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score


In [ ]:
def extract_features(audio_path, sr=16000):

    signal, sr = librosa.load(audio_path, sr=sr)

    # =====================
    # MFCC (13)
    # =====================
    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=13
    )
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)
    mfcc_min = np.min(mfcc, axis=1)
    mfcc_max = np.max(mfcc, axis=1)

    # =====================
    # PLP aproximado manual
    # =====================

    # Espectrograma Mel
    mel_spec = librosa.feature.melspectrogram(
        y=signal,
        sr=sr,
        n_mels=26,
        power=2
    )

    # Compresión log
    log_mel = np.log(mel_spec + 1e-9)

    # DCT (similar a PLP)
    plp = librosa.feature.mfcc(
        S=log_mel,
        n_mfcc=13
    )

    plp_mean = np.mean(plp, axis=1)
    plp_std = np.std(plp, axis=1)
    plp_min = np.min(plp, axis=1)
    plp_max = np.max(plp, axis=1)

    # Concatenar
    features = np.concatenate([
    mfcc_mean, mfcc_std, mfcc_min, mfcc_max,
    plp_mean, plp_std, plp_min, plp_max
    ])

    return features


In [ ]:
emotion_map = {
    "alegria": "positivo",
    "neutro": "neutral",
    "tristeza": "negativo",
    "miedo": "negativo",
    "ira": "negativo",
    "asco": "negativo",
    "sorpresa": "negativo"

}

emotion2id = {
    "positivo": 0,
    "neutral": 1,
    "negativo": 2
}

def load_dataset(dataset_path):
    X, y = [], []

    for file in os.listdir(dataset_path):
        if not file.lower().endswith(".wav"):
            continue

        # Obtener emoción (segunda parte del nombre)
        emotion_raw = file.split("_")[1].lower()

        if emotion_raw not in emotion_map:
            continue

        label = emotion2id[emotion_map[emotion_raw]]

        features = extract_features(os.path.join(dataset_path, file))

        X.append(features)
        y.append(label)

    return np.array(X), np.array(y)


In [ ]:
DATASET_PATH = "/content/dataset"

X, y = load_dataset(DATASET_PATH)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')
f1 = f1_score(y_test, y_pred, average='macro')

print("===== MÉTRICAS GLOBALES =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print(classification_report(
    y_test,
    y_pred,
    target_names=["positivo", "neutral", "negativo"]
))

# ============================
# BASELINE ALEATORIA
# ============================

print("\n===== BASELINE ALEATORIA =====")

# Fijar semilla para reproducibilidad
np.random.seed(42)

# Generar predicciones aleatorias (3 clases: 0,1,2)
random_preds = np.random.randint(0, 3, size=len(y_test))

# Métricas baseline
rand_accuracy = accuracy_score(y_test, random_preds)
rand_precision = precision_score(y_test, random_preds, average='macro', zero_division=0)
rand_recall = recall_score(y_test, random_preds, average='macro', zero_division=0)
rand_f1 = f1_score(y_test, random_preds, average='macro', zero_division=0)

print(f"Accuracy : {rand_accuracy:.4f}")
print(f"Precision: {rand_precision:.4f}")
print(f"Recall   : {rand_recall:.4f}")
print(f"F1-score : {rand_f1:.4f}")

print("\nReporte baseline:\n")
print(classification_report(
    y_test,
    random_preds,
    target_names=["positivo", "neutral", "negativo"]
))



===== MÉTRICAS GLOBALES =====
Accuracy : 0.4638
Precision: 0.5118
Recall   : 0.5992
F1-score : 0.4581
              precision    recall  f1-score   support

    positivo       0.24      0.67      0.35        51
     neutral       0.47      0.85      0.61        95
    negativo       0.83      0.28      0.42       255

    accuracy                           0.46       401
   macro avg       0.51      0.60      0.46       401
weighted avg       0.67      0.46      0.45       401


===== BASELINE ALEATORIA =====
Accuracy : 0.3142
Precision: 0.3099
Recall   : 0.3122
F1-score : 0.2815

Reporte baseline:

              precision    recall  f1-score   support

    positivo       0.12      0.31      0.17        51
     neutral       0.23      0.31      0.26        95
    negativo       0.58      0.32      0.41       255

    accuracy                           0.31       401
   macro avg       0.31      0.31      0.28       401
weighted avg       0.44      0.31      0.35       401



In [ ]:
#Carga del dataset
DATASET_PATH = "/content/dataset"

X, y = load_dataset(DATASET_PATH)

print("Muestras totales:", X.shape)

Muestras totales: (2005, 104)


In [ ]:
#Division de train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
#Modelo SVM
model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale",
    class_weight="balanced"
)

model.fit(X_train, y_train)

SVC(C=10, class_weight='balanced')

In [ ]:
#Evaliacion
y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nReporte de clasificación:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=["positivo", "neutral", "negativo"]
))


Accuracy: 0.46384039900249374

Reporte de clasificación:

              precision    recall  f1-score   support

    positivo       0.24      0.67      0.35        51
     neutral       0.47      0.85      0.61        95
    negativo       0.83      0.28      0.42       255

    accuracy                           0.46       401
   macro avg       0.51      0.60      0.46       401
weighted avg       0.67      0.46      0.45       401

